# Dose Whisperer — Notebook 2b: Synthetic errors, labels and grouped train/val/test split

> **Research prototype.** Labels are *prototype safety flags* (SAFE / WARNING / HIGH), not medical diagnoses.
> MIMIC-III Demo prescriptions are **not assumed to be clinically safe**: the original prescriptions are labelled by the same rules as the perturbed ones (`error_type = none`).

**What this notebook produces** (in `data/splits/`): `train.csv`, `val.csv`, `test.csv`, `evidence.csv`, `split_report.json`, `examples_preview.csv` — exactly the files and columns Notebook 3 expects (`TRAIN_PATH`, `VAL_PATH`, `TEST_PATH`, `EVIDENCE_PATH`).

**Input table:** the notebook looks for the cleaned prescription table in this order: (1) a `canonical_df` already in the kernel, (2) `data/canonical_medications.csv`, (3) **it rebuilds the table from the raw MIMIC files** (same steps as Notebook 2: adult filter, drug-name fallback, dose parsing, unit and route normalisation) and saves it to `data/canonical_medications.csv`. So you do not need Notebook 2's kernel.

**Design decisions (please read):**
1. **Frequency does not exist in MIMIC-III PRESCRIPTIONS**, so each original prescription is given a *standard frequency* from `FREQ_RULES` (edit it — it must come from the drug label), and frequency errors perturb that.
2. **Dose / unit / frequency labels** come from the **verified `dose_table.json`** (Notebook 1) plus `FREQ_RULES` — *not* from regex over label text. This keeps the rules-only baseline in Notebook 3 a genuine baseline.
3. **Condition-risk labels** come from the retrieved openFDA text: a condition is HIGH if a synonym appears in a *contraindication* snippet, WARNING if in a *warning/precaution* snippet. Only the text the model will actually see (same snippet caps as Notebook 3) is used, so a positive label is always supported by visible evidence. *Caveat: this resembles Notebook 3's rules-only condition matcher, so that baseline will look strong on condition risk — say so in the report.*
4. **Conditions** are real (MIMIC ICD-9 diagnoses per admission, mapped to a curated list) **plus synthetic condition contexts** (flagged in `condition_source`) because 100 demo patients will not naturally cover the drug-condition pairs. Report this honestly.
5. **No leakage:** the split is made **by `subject_id`** (stricter than by prescription): no patient, admission or original prescription appears in two splits.

In [ ]:
# ===== 1. CONFIG (edit this cell) =====
from pathlib import Path
import json, re, math, random, itertools
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

PROJECT_DIR     = Path("/home/user9/SOURABH_S")
MIMIC_DIR       = Path("/home/user9/dataset")                 # raw MIMIC-III Demo CSVs
DATA_DIR        = PROJECT_DIR / "data"
DOSE_TABLE_PATH = DATA_DIR / "dose_table.json"
LABEL_CACHE_DIR = DATA_DIR / "label_cache"                    # openFDA label JSONs from Notebook 1
CANONICAL_PATH  = DATA_DIR / "canonical_medications.csv"      # saved at the end of Notebook 2
OUT_DIR         = DATA_DIR / "splits"

SEED = 42
MAX_DRUGS = 15                    # curated list size (top-N most frequent drugs that have BOTH a verified dose limit and label evidence)
MAX_ORIGINALS_PER_DRUG = 150      # cap per drug to keep the set balanced
VARIANTS_PER_ORIGINAL = 4         # perturbed copies per original prescription (plus the unmodified original)
N_SYNTHETIC_CONTEXTS = 2          # extra synthetic condition contexts per original (0 disables); 1 relevant + 1 irrelevant
ALLOWED_ROUTES = None             # e.g. {"ORAL","IV","IV_BOLUS"}; None = all
SPLIT_FRACTIONS = dict(train=0.70, val=0.15, test=0.15)
USE_LIVE_OPENFDA_FETCH = False    # True => fetch labels for drugs missing from label_cache (needs internet; results cached to label_cache)

# Keep IDENTICAL to Notebook 3) so that labels only use evidence the model can see
SNIPPET_CHARS, MAX_SNIPPETS, SECTION_CHARS = 350, 8, 1800

# Severity thresholds: keep IDENTICAL to Notebook 3's CHK dict so labels and the deterministic checker agree on severity
CHK = dict(over_high=2.0, under_high=5.0, daily_high=2.0, unit_high=10.0)

# Standard frequency tokens and doses per day (tokens are all parsed by Notebook 3)
FREQ_PER_DAY = {"daily": 1, "BID": 2, "TID": 3, "QID": 4, "q8h": 3, "q6h": 4, "q4h": 6, "q2h": 12, "q1h": 24}

# ---- FREQ_RULES: (standard_frequency, min_doses_per_day, max_doses_per_day) per drug (substring match on the drug key). ----
# !!! EXAMPLES ONLY - verify every entry against the label text / your verified table. Drugs not listed use FREQ_DEFAULT. !!!
FREQ_RULES = {
    "acetaminophen": ("q6h", 4, 6),
    "furosemide":    ("daily", 1, 2),
    "metoprolol":    ("BID", 1, 2),
    "lorazepam":     ("BID", 2, 3),
    "warfarin":      ("daily", 1, 1),
    "heparin":       ("q8h", 2, 3),
}
FREQ_DEFAULT = ("BID", 1, 2)

# ---- column names inside dose_table.json entries: candidates are tried in order (edit if your keys differ) ----
DT_KEYS = {
    "drug": "drug",
    "route": "route",
    "scope": "scope",
    "unit": "unit",
    "field": "field",
    "value": "value",
}

# ---- curated conditions: ICD-9 prefixes (MIMIC icd9_code has no dots) and the text synonyms looked for in label snippets ----
COND = {
 "renal impairment":         dict(icd=["584", "585", "586", "403", "404"], syn=["renal", "kidney", "nephro", "creatinine", "egfr", "dialysis"]),
 "hepatic impairment":       dict(icd=["570", "571", "572", "573"],        syn=["hepatic", "liver", "cirrhosis", "hepatitis"]),
 "heart failure":            dict(icd=["428"],                              syn=["heart failure", "cardiac failure", "congestive"]),
 "peptic ulcer disease":     dict(icd=["531", "532", "533", "534"],         syn=["ulcer", "peptic"]),
 "gastrointestinal bleeding":dict(icd=["578"],                              syn=["gastrointestinal bleed", "gi bleed", "hemorrhag"]),
 "asthma":                   dict(icd=["493"],                              syn=["asthma", "bronchospasm"]),
 "COPD":                     dict(icd=["491", "492", "496"],                syn=["copd", "chronic obstructive"]),
 "diabetes mellitus":        dict(icd=["250"],                              syn=["diabet", "hypoglyc"]),
 "hypertension":             dict(icd=["401"],                              syn=["hypertens"]),
 "arrhythmia":               dict(icd=["4273", "4271", "4272"],             syn=["arrhythm", "fibrillation"]),
 "seizure disorder":         dict(icd=["345"],                              syn=["seizure", "epilep"]),
 "bleeding disorder":        dict(icd=["286", "2875"],                      syn=["bleeding", "coagulopathy", "thrombocytopenia"]),
 "hyperkalemia":             dict(icd=["2767"],                             syn=["hyperkalemia"]),
 "hypokalemia":              dict(icd=["2768"],                             syn=["hypokalemia"]),
 "pregnancy":                dict(icd=[],                                   syn=["pregnan", "fetal"]),   # synthetic contexts only
}
print("config OK | out dir:", OUT_DIR)

config OK | out dir: /home/user9/SOURABH_S/data/splits


In [6]:
# ===== 2. LOAD THE VERIFIED DOSE TABLE (Notebook 1) =====
MASS = {"mg": 1.0, "mcg": 1e-3, "g": 1e3}
def norm_unit(u):
    s = str(u or "").strip().lower().replace("µ", "u").replace("μ", "u")
    return {"milligram": "mg", "milligrams": "mg", "microgram": "mcg", "micrograms": "mcg", "ug": "mcg", "gm": "g", "gram": "g", "grams": "g",
            "units": "unit", "u": "unit"}.get(s, s)

def pick(d, names):
    for n in names:
        if n in d and d[n] not in (None, "", "NA", "N/A"):
            return d[n]
    return None

def to_float(x):
    try: return float(str(x).replace(",", ""))
    except Exception: return None

def norm_name(s):
    s = re.sub(r"\*nf\*", " ", str(s).lower())
    s = re.sub(r"\([^)]*\)", " ", s)
    return " ".join(re.sub(r"[^a-z0-9]+", " ", s).split())

raw = json.loads(DOSE_TABLE_PATH.read_text())
entries = raw.get("limits", raw) if isinstance(raw, dict) else raw
if isinstance(entries, dict): entries = [dict(v, drug=k) if isinstance(v, dict) else {"drug": k} for k, v in entries.items()]
print("entries in dose_table:", len(entries))
print("keys of first entry  :", list(entries[0].keys()) if entries else None)
print("first entry          :", entries[0] if entries else None)

DOSE_REF, skipped = {}, []
DOSE_REF = {}
skipped = []

for e in entries:
    drug = e.get("drug")
    route = e.get("route")
    scope = e.get("scope")
    unit = e.get("unit")
    field = e.get("field")
    value = e.get("value")

    # Ignore incomplete/non-dose entries
    if not drug or not unit or scope != "dose" or value is None:
        skipped.append((drug, "incomplete/non-dose entry"))
        continue

    try:
        value = float(value)
    except (TypeError, ValueError):
        skipped.append((drug, "non-numeric value"))
        continue

    key = (drug.strip().lower(), route.strip().upper() if route else None)

    if key not in DOSE_REF:
        DOSE_REF[key] = {
            "lo": None,
            "hi": None,
            "max_daily": None,
            "unit": unit
        }

    if field == "usual_low":
        DOSE_REF[key]["lo"] = value

    elif field == "usual_high":
        DOSE_REF[key]["hi"] = value

print("usable dose references:", len(DOSE_REF))

for k, v in list(DOSE_REF.items())[:10]:
    print(k, "->", v)

print("skipped:", len(skipped))
print(f"\nusable verified limits: {len(DOSE_REF)} | skipped: {len(skipped)}")
for s in skipped[:10]: print("  skipped:", s)
if not DOSE_REF:
    raise RuntimeError("No usable entries. Edit DT_KEYS (section 1) so the key names match the printed 'keys of first entry'.")
print("drugs with no min single dose (under-dose labels impossible):", [k for k, v in DOSE_REF.items() if v["lo"] is None][:15])

entries in dose_table: 34
keys of first entry  : ['drug', 'route', 'scope', 'unit', 'field', 'value', 'labels_in_group', 'labels_with_quote', 'pct', 'status', 'quote']
first entry          : {'drug': 'furosemide', 'route': 'ORAL', 'scope': 'dose', 'unit': 'mg', 'field': 'usual_low', 'value': 20, 'labels_in_group': 97, 'labels_with_quote': 63, 'pct': 0.65, 'status': 'OK', 'quote': 'usual initial dose of furosemide tablets is 20 to 80 mg given as a single dose'}
usable dose references: 8
('furosemide', 'ORAL') -> {'lo': 20.0, 'hi': 80.0, 'max_daily': None, 'unit': 'mg'}
('furosemide', 'INTRAVENOUS') -> {'lo': 20.0, 'hi': 40.0, 'max_daily': None, 'unit': 'mg'}
('metoprolol', 'INTRAVENOUS') -> {'lo': None, 'hi': 5.0, 'max_daily': None, 'unit': 'mg'}
('vancomycin', 'INTRAVENOUS') -> {'lo': 500.0, 'hi': 1000.0, 'max_daily': None, 'unit': 'mg'}
('lorazepam', 'INTRAVENOUS') -> {'lo': None, 'hi': 4.0, 'max_daily': None, 'unit': 'mg'}
('potassium_chloride', 'ORAL') -> {'lo': None, 'hi': 20.0, 'm

In [ ]:
# ============================================================
# 3. OPENFDA EVIDENCE PER DRUG
#    Adapted for the existing label_cache format
# ============================================================

SECTION_FIELDS = {
    "dosage": [
        "dosage_and_administration",
        "dosage_forms_and_strengths",
        "dosage"
    ],

    "warnings": [
        "boxed_warning",
        "warnings",
        "warnings_and_cautions",
        "precautions",
        "general_precautions"
    ],

    "contra": [
        "contraindications",
        "contra"
    ],
}


def _txt(v):
    """Convert cached section data into text."""

    if v is None:
        return ""

    if isinstance(v, list):
        return "\n".join(str(x) for x in v)

    if isinstance(v, dict):
        return "\n".join(
            str(x) for x in v.values()
        )

    return str(v)


def doc_to_sections(doc):
    """
    Extract dosage, warnings and contraindications
    from the cached document.
    """

    # Your cache stores the actual label sections here
    sections = doc.get("sections", {})

    if not isinstance(sections, dict):
        sections = {}

    output = {}

    for sec, fields in SECTION_FIELDS.items():

        pieces = []

        for field in fields:

            # First check inside "sections"
            value = sections.get(field)

            # Also support fields directly on document
            if value is None:
                value = doc.get(field)

            text = _txt(value)

            if text.strip():
                pieces.append(text)

        output[sec] = "\n".join(pieces)

    return output


def candidate_docs(obj):
    """
    Handle the actual label_cache format.

    Your files are lists of dictionaries.
    """

    if isinstance(obj, list):

        return [
            r for r in obj
            if isinstance(r, dict)
        ]

    if isinstance(obj, dict):

        # Raw OpenFDA response
        if isinstance(obj.get("results"), list):

            return [
                r for r in obj["results"]
                if isinstance(r, dict)
            ]

        # Single cached document
        if (
            "sections" in obj  or "generic_name" in obj  or "brand_name" in obj ):
            return [obj]

    return []

def doc_names(doc, fname):
    """
    Get drug names from the cached document.
    """

    names = []

    for field in (
        "generic_name",  "substance_name", "brand_name"):

        value = doc.get(field, [])

        if isinstance(value, str):
            value = [value]

        if isinstance(value, list):

            for name in value:

                if name is not None:

                    names.append(norm_name(str(name)) )

    # Filename is also useful
    names.append( norm_name(  Path(fname).stem ))

    return names

# ============================================================
# LOAD EVIDENCE
# ============================================================

EVIDENCE = {}
seen_files = 0
matched_docs = 0


if LABEL_CACHE_DIR.exists():

    for f in sorted( LABEL_CACHE_DIR.rglob("*.json")  ):

        try:
            obj = json.loads( f.read_text() )
        except Exception as e:
            print( "Could not read:", f.name, repr(e) )
            continue
        docs = candidate_docs(obj)
        seen_files += 1
        # ----------------------------------------------------
        # Examine every cached label
        # ----------------------------------------------------

        for doc in docs:

            names_raw = doc_names( doc, f.name)

            names = [
                str(x).strip().lower()
                for x in names_raw
                if x is not None
            ]

            # Combine names
            names_text = " ".join(names)

            names_words = set(
                names_text
                .replace("_", " ")
                .split()            )

            # ------------------------------------------------
            # Compare against DOSE_REF
            # ------------------------------------------------

            for dose_key in DOSE_REF:

                # DOSE_REF keys are tuples:
                # ('furosemide', 'ORAL')
                drug_name = str( dose_key[0] ).strip().lower()

                route_needed = str(   dose_key[1]).strip().upper()
                # --------------------------------------------
                # Drug name matching
                # --------------------------------------------

                drug_words = set(
                    drug_name
                    .replace("_", " ")
                    .split()  )

                name_match = ( drug_words <= names_words)
                filename = f.stem.lower()
                filename_match = (
                    drug_name.replace(
                        " ",
                        "_"
                    )   in filename      )


                if not ( name_match  or filename_match ):
                    continue

                # --------------------------------------------
                # Route matching
                # --------------------------------------------

                doc_route = str(      doc.get(
                        "route",
                        ""                    )
                ).upper()

                route_match = (
                    route_needed
                    in doc_route
                    or doc_route
                    in route_needed
                )


                # If route is missing, don't reject it
                # because some cached labels may not contain it.
                if (
                    doc_route
                    and not route_match
                ):
                    continue
                # --------------------------------------------
                # Extract evidence
                # --------------------------------------------

                sec = doc_to_sections( doc )
                has_evidence = any(
                    sec.get(section, "").strip()
                    for section in (
                        "dosage",
                        "warnings",
                        "contra"     ))

                if not has_evidence:
                    continue

               # --------------------------------------------
                # Store evidence
                # --------------------------------------------

                matched_docs += 1
                if dose_key not in EVIDENCE:

                    EVIDENCE[dose_key] = {
                        "dosage": "",
                        "warnings": "",
                        "contra": "",
                        "_n": 0
                    }

                for section in ( "dosage", "warnings",     "contra"  ):

                    text = sec.get(
                        section,
                        ""
                    ).strip()

                    if text:

                        # Avoid duplicate evidence
                        if text not in EVIDENCE[
                            dose_key
                        ][section]:

                            existing = EVIDENCE[
                                dose_key
                            ][section]

                            if existing:
                                EVIDENCE[
                                    dose_key
                                ][section] += (
                                    "\n" + text
                                )
                            else:
                                EVIDENCE[
                                    dose_key
                                ][section] = text


                EVIDENCE[
                    dose_key
                ]["_n"] += 1


print(
    f"label_cache JSON files read: "
    f"{seen_files}")

print( f"matching label documents: " f"{matched_docs}")

print(
    f"drugs/routes with evidence: "
    f"{len(EVIDENCE)} / {len(DOSE_REF)}")


# ============================================================
# SHOW WHAT WE FOUND
# ============================================================

for key, value in EVIDENCE.items():

    print(
        f"\n{key}: "
        f"D={len(value['dosage'])} "
        f"W={len(value['warnings'])} "
        f"C={len(value['contra'])} "
        f"labels={value['_n']}"    )


# ============================================================
# MISSING EVIDENCE
# ============================================================

missing = [
    k
    for k in DOSE_REF
    if k not in EVIDENCE
]

if missing:

    print(
        "\n⚠️ Drugs/routes with NO evidence:"  )

    for item in missing:
        print(
            "  ",
            item        )
if not EVIDENCE:
    raise RuntimeError(
        "No evidence loaded from label_cache. "
        "The cache structure or matching logic "
        "needs further inspection."    )

# ============================================================
# SENTENCE SNIPPETS
# ============================================================

def _clean(s):

    return re.sub(
        r"\s+",
        " ",
        str(s or "") ).strip()


_SENT = re.compile(
    r"(?<=[.;!?])\s+(?=[A-Z0-9(\[])")


def split_snippets(
    text,
    snippet_chars=SNIPPET_CHARS,
    max_snips=MAX_SNIPPETS,
    section_chars=SECTION_CHARS
):

    parts = []
    for block in re.split(
        r"\n+",
        str(text or "") ):

        parts.extend(
            _SENT.split(
                _clean(block)    )     )


    out = []
    seen = set()
    used = 0


    for p in parts:
        p = _clean(p)
        if len(p) < 12:
            continue
        if p.lower() in seen:
            continue
        seen.add(p.lower())

        p = p[
            :snippet_chars
        ].rstrip()


        if (  used + len(p)  > section_chars and out):
            break

        out.append(p)

        used += len(p)
        if len(out) >= max_snips:
            break
    return out

# ============================================================
# VISIBLE EVIDENCE
# ============================================================

VISIBLE = {

    k: {
        sec: split_snippets( v.get(sec, "") )
        for sec in ("dosage", "warnings", "contra"  )
    }
    for k, v in EVIDENCE.items()
}

print("\nVisible snippets per drug/route (D/W/C):")
for k in VISIBLE:

    print(
        k,
        tuple(
            len(VISIBLE[k][s])
            for s in (
                "dosage",
                "warnings",
                "contra"
            )
        )
    )

label_cache JSON files read: 12
matching label documents: 506
drugs/routes with evidence: 8 / 8

('acetaminophen', 'INTRAVENOUS'): D=119463 W=76542 C=5713 labels=21

('furosemide', 'ORAL'): D=66402 W=367210 C=562 labels=97

('furosemide', 'INTRAVENOUS'): D=92542 W=428298 C=995 labels=49

('lorazepam', 'INTRAVENOUS'): D=43798 W=319175 C=2676 labels=9

('magnesium_sulfate', 'INTRAVENOUS'): D=106519 W=167185 C=1394 labels=33

('metoprolol', 'INTRAVENOUS'): D=25270 W=85283 C=8317 labels=22

('potassium_chloride', 'ORAL'): D=294254 W=324604 C=25755 labels=219

('vancomycin', 'INTRAVENOUS'): D=440217 W=461175 C=2144 labels=56

Visible snippets per drug/route (D/W/C):
('acetaminophen', 'INTRAVENOUS') (8, 8, 8)
('furosemide', 'ORAL') (8, 8, 4)
('furosemide', 'INTRAVENOUS') (8, 8, 6)
('lorazepam', 'INTRAVENOUS') (8, 8, 7)
('magnesium_sulfate', 'INTRAVENOUS') (8, 8, 7)
('metoprolol', 'INTRAVENOUS') (8, 8, 8)
('potassium_chloride', 'ORAL') (8, 8, 8)
('vancomycin', 'INTRAVENOUS') (8, 8, 8)


In [18]:
# ===== ADAPTER: route-aware DOSE_REF / EVIDENCE -> string keys like "furosemide (ORAL)" =====
MASS = {"mg": 1.0, "mcg": 1e-3, "g": 1e3}
def norm_unit(u):
    s = str(u or "").strip().lower().replace("µ", "u").replace("μ", "u")
    return {"milligram": "mg", "milligrams": "mg", "microgram": "mcg", "micrograms": "mcg", "ug": "mcg",
            "gm": "g", "gram": "g", "grams": "g", "units": "unit", "u": "unit"}.get(s, s)
def norm_name(s):
    s = re.sub(r"\*nf\*", " ", str(s).lower()); s = re.sub(r"\([^)]*\)", " ", s)
    return " ".join(re.sub(r"[^a-z0-9]+", " ", s).split())
def _txt(v):
    if v is None: return ""
    return "\n".join(str(x) for x in v) if isinstance(v, list) else str(v)
def _clean(s): return re.sub(r"\s+", " ", str(s or "")).strip()
_SENT = re.compile(r"(?<=[.;!?])\s+(?=[A-Z0-9(\[])")
def split_snippets(text, snippet_chars=SNIPPET_CHARS, max_snips=MAX_SNIPPETS, section_chars=SECTION_CHARS):
    parts = []
    for block in re.split(r"\n+", str(text or "")): parts.extend(_SENT.split(_clean(block)))
    out, seen, used = [], set(), 0
    for p in parts:
        p = _clean(p)
        if len(p) < 12 or p.lower() in seen: continue
        seen.add(p.lower()); p = p[:snippet_chars].rstrip()
        if used + len(p) > section_chars and out: break
        out.append(p); used += len(p)
        if len(out) >= max_snips: break
    return out

def disp(name, route): return f"{str(name).replace('_', ' ')} ({route})"

def to_sections(v):
    # turns one stored evidence value into dosage / warnings / contra text (handles dicts, openFDA 'results' wrappers, plain strings)
    if isinstance(v, dict) and isinstance(v.get("results"), list) and v["results"]: v = v["results"][0]
    out = {"dosage": "", "warnings": "", "contra": ""}
    if isinstance(v, dict):
        for kk, vv in v.items():
            kl, t = str(kk).lower(), _txt(vv)
            if not t.strip(): continue
            if "contra" in kl: out["contra"] += "\n" + t
            elif "dos" in kl: out["dosage"] += "\n" + t
            elif any(w in kl for w in ("warn", "precaut", "boxed")): out["warnings"] += "\n" + t
    elif isinstance(v, str): out["dosage"] = v
    return {k: x.strip() for k, x in out.items()}

if any(isinstance(k, tuple) for k in DOSE_REF):
    RAW_DOSE_REF, RAW_EVIDENCE = DOSE_REF, EVIDENCE
    DOSE_REF = {disp(*k): dict(v, unit=norm_unit(v["unit"]), drug=str(k[0]).replace("_", " "), route=k[1], key=disp(*k)) for k, v in RAW_DOSE_REF.items()}
    EVIDENCE = {disp(*k): to_sections(v) for k, v in RAW_EVIDENCE.items()}
    print("converted tuple keys -> strings")
EVIDENCE = {k: v for k, v in EVIDENCE.items() if any(v.values())}
VISIBLE = {k: {s: split_snippets(v[s]) for s in ("dosage", "warnings", "contra")} for k, v in EVIDENCE.items()}

print("DOSE_REF entries:", list(DOSE_REF))
print("with evidence   :", [k for k in DOSE_REF if k in EVIDENCE])
print("NO evidence     :", [k for k in DOSE_REF if k not in EVIDENCE])
print("visible snippets (D/W/C):", {k: tuple(len(VISIBLE[k][s]) for s in ("dosage", "warnings", "contra")) for k in VISIBLE})
if not EVIDENCE:
    print("Evidence sections came out empty. Structure of one stored value:")
    k0 = next(iter(RAW_EVIDENCE)); v0 = RAW_EVIDENCE[k0]
    print(k0, type(v0), list(v0.keys())[:20] if isinstance(v0, dict) else str(v0)[:300])

converted tuple keys -> strings
DOSE_REF entries: ['furosemide (ORAL)', 'furosemide (INTRAVENOUS)', 'metoprolol (INTRAVENOUS)', 'vancomycin (INTRAVENOUS)', 'lorazepam (INTRAVENOUS)', 'potassium chloride (ORAL)', 'magnesium sulfate (INTRAVENOUS)', 'acetaminophen (INTRAVENOUS)']
with evidence   : ['furosemide (ORAL)', 'furosemide (INTRAVENOUS)', 'metoprolol (INTRAVENOUS)', 'vancomycin (INTRAVENOUS)', 'lorazepam (INTRAVENOUS)', 'potassium chloride (ORAL)', 'magnesium sulfate (INTRAVENOUS)', 'acetaminophen (INTRAVENOUS)']
NO evidence     : []
visible snippets (D/W/C): {'acetaminophen (INTRAVENOUS)': (8, 8, 8), 'furosemide (ORAL)': (8, 8, 4), 'furosemide (INTRAVENOUS)': (8, 8, 6), 'lorazepam (INTRAVENOUS)': (8, 8, 7), 'magnesium sulfate (INTRAVENOUS)': (8, 8, 7), 'metoprolol (INTRAVENOUS)': (8, 8, 8), 'potassium chloride (ORAL)': (8, 8, 8), 'vancomycin (INTRAVENOUS)': (8, 8, 8)}


In [19]:
# ===== 4. REAL PATIENT CONDITIONS from MIMIC ICD-9 diagnoses =====
dx = pd.read_csv(MIMIC_DIR / "DIAGNOSES_ICD.csv", low_memory=False); dx.columns = [c.lower() for c in dx.columns]
dx["icd9_code"] = dx["icd9_code"].astype(str).str.strip().str.upper()
def icd_to_conditions(code):
    return [c for c, d in COND.items() if any(code.startswith(p) for p in d["icd"])]
dx["conds"] = dx["icd9_code"].map(icd_to_conditions)
REAL_CONDS = (dx[dx["conds"].map(len) > 0].explode("conds").groupby("hadm_id")["conds"].agg(lambda s: sorted(set(s))).to_dict())
print("admissions with >=1 mapped condition:", len(REAL_CONDS), "of", dx["hadm_id"].nunique())
print("condition frequency (real):", dict(Counter(c for v in REAL_CONDS.values() for c in v).most_common()))

admissions with >=1 mapped condition: 119 of 129
condition frequency (real): {'renal impairment': 62, 'hypertension': 55, 'arrhythmia': 50, 'diabetes mellitus': 48, 'heart failure': 40, 'COPD': 18, 'hepatic impairment': 14, 'bleeding disorder': 13, 'gastrointestinal bleeding': 10, 'hypokalemia': 7, 'asthma': 6, 'peptic ulcer disease': 6, 'seizure disorder': 5, 'hyperkalemia': 5}


In [22]:
# ===== 5. SELECT ELIGIBLE ORIGINAL PRESCRIPTIONS =====
def build_canonical(mimic_dir):
    # Rebuilds Notebook 2's canonical medication table from raw MIMIC-III Demo files.
    rx = pd.read_csv(mimic_dir / "PRESCRIPTIONS.csv", low_memory=False); rx.columns = [c.lower() for c in rx.columns]
    pt = pd.read_csv(mimic_dir / "PATIENTS.csv", low_memory=False);      pt.columns = [c.lower() for c in pt.columns]
    ad = pd.read_csv(mimic_dir / "ADMISSIONS.csv", low_memory=False);    ad.columns = [c.lower() for c in ad.columns]
    pt["dob"] = pd.to_datetime(pt["dob"], errors="coerce"); ad["admittime"] = pd.to_datetime(ad["admittime"], errors="coerce")
    a = ad[["subject_id", "hadm_id", "admittime"]].merge(pt[["subject_id", "dob"]].drop_duplicates("subject_id"), on="subject_id", how="left")
    a["age_years"] = a["admittime"].dt.year - a["dob"].dt.year                      # year arithmetic avoids timedelta overflow on shifted dates
    before_bd = (a["admittime"].dt.month < a["dob"].dt.month) | ((a["admittime"].dt.month == a["dob"].dt.month) & (a["admittime"].dt.day < a["dob"].dt.day))
    a.loc[before_bd, "age_years"] -= 1
    a["age_deidentified"] = a["age_years"] > 100                                     # MIMIC shifts DOB of patients over 89 -> age ~300
    a = a[(a["age_years"] >= 18) | a["age_deidentified"]]
    df = rx.merge(a[["subject_id", "hadm_id", "age_years", "age_deidentified"]], on=["subject_id", "hadm_id"], how="inner")
    df["drug_raw"] = df["drug"].astype(str).str.strip()
    df["drug_name_normalized"] = df["drug_name_generic"].fillna(df["drug"]).astype(str).str.strip()
    bad = df["drug_name_normalized"].isin(["", "nan", "<MISSING>"]); df.loc[bad, "drug_name_normalized"] = df.loc[bad, "drug_raw"]
    def parse(v):
        if pd.isna(v) or str(v).strip() == "": return np.nan, np.nan, "missing"
        t = str(v).strip().replace(",", "")
        try: x = float(t); return x, x, "single"
        except ValueError: pass
        m = re.fullmatch(r"\s*(\d+(?:\.\d+)?)\s*-\s*(\d+(?:\.\d+)?)\s*", t)
        if m and float(m.group(1)) <= float(m.group(2)): return float(m.group(1)), float(m.group(2)), "range"
        return np.nan, np.nan, "unparsed"
    p = df["dose_val_rx"].apply(parse)
    df["dose_min"], df["dose_max"], df["dose_type"] = p.apply(lambda x: x[0]), p.apply(lambda x: x[1]), p.apply(lambda x: x[2])
    umap = {"ml": "mL", "gm": "g", "UNIT": "unit", "million units": "million_unit", "mg PE": "mg_PE"}
    df["dose_unit_normalized"] = df["dose_unit_rx"].astype(str).str.strip().map(lambda u: umap.get(u, u))
    rmap = {"PO": "ORAL", "ORAL": "ORAL", "IV": "IV", "IV DRIP": "IV_DRIP", "IV BOLUS": "IV_BOLUS", "IVPCA": "IV_PCA", "SC": "SUBCUTANEOUS",
            "SUBCUT": "SUBCUTANEOUS", "PO/NG": "ORAL_OR_NG", "NG": "NASOGASTRIC", "OG": "OROGASTRIC", "J TUBE": "JEJUNAL_TUBE", "IM": "INTRAMUSCULAR",
            "PR": "RECTAL", "IT": "INTRATHECAL", "TP": "TOPICAL", "TD": "TRANSDERMAL", "SL": "SUBLINGUAL", "IH": "INHALATION", "NEB": "NEBULIZED",
            "DIALYS": "DIALYSIS", "OD": "RIGHT_EYE", "OS": "LEFT_EYE", "OU": "BOTH_EYES", "RIGHT EYE": "RIGHT_EYE", "LEFT EYE": "LEFT_EYE", "BOTH EYES": "BOTH_EYES"}
    df["route_normalized"] = df["route"].astype(str).str.strip().map(lambda r: rmap.get(r, r))
    df["verification_ready"] = df["drug_name_normalized"].notna() & df["dose_min"].notna() & df["dose_max"].notna() & df["dose_unit_normalized"].notna() & df["route_normalized"].notna()
    cols = ["subject_id", "hadm_id", "age_years", "age_deidentified", "drug_name_normalized", "drug_raw", "dose_min", "dose_max", "dose_type",
            "dose_unit_normalized", "route_normalized", "verification_ready"]
    return df[cols].copy()

try:
    base = canonical_df.copy(); print("using canonical_df from the running kernel")
except NameError:
    if CANONICAL_PATH.exists():
        base = pd.read_csv(CANONICAL_PATH, low_memory=False); print("loaded", CANONICAL_PATH)
    else:
        print("canonical table not found -> rebuilding from raw MIMIC files in", MIMIC_DIR)
        base = build_canonical(MIMIC_DIR); CANONICAL_PATH.parent.mkdir(parents=True, exist_ok=True)
        base.to_csv(CANONICAL_PATH, index=False); print("saved", CANONICAL_PATH)
print("canonical rows:", len(base), "| expected for the MIMIC-III Demo: about 10,256")
base = base.reset_index(drop=True)

ROUTE_TO_TABLE = {"ORAL": "ORAL", "ORAL_OR_NG": "ORAL", "IV": "INTRAVENOUS", "IV_BOLUS": "INTRAVENOUS",
                  "IV_DRIP": "INTRAVENOUS", "IV_PCA": "INTRAVENOUS", "SUBCUTANEOUS": "SUBCUTANEOUS", "INTRAMUSCULAR": "INTRAMUSCULAR"}
entries_sorted = sorted([k for k in DOSE_REF if k in EVIDENCE], key=lambda k: -len(DOSE_REF[k]["drug"]))
def match_key(name, route):
    toks, r = set(norm_name(name).split()), ROUTE_TO_TABLE.get(route)
    for k in entries_sorted:
        d = DOSE_REF[k]
        if d["route"] == r and set(norm_name(d["drug"]).split()) <= toks: return k
    return None

base["drug_key"] = [match_key(n, r) for n, r in zip(base["drug_name_normalized"], base["route_normalized"])]

#base["drug_key"] = base["drug_name_normalized"].map(match_key)
n0 = len(base)
steps = [("verification_ready", base["verification_ready"].astype(bool)),
         ("single numeric dose", (base["dose_type"] == "single") & (pd.to_numeric(base["dose_min"], errors="coerce") > 0)),
         ("matched to a verified drug with evidence", base["drug_key"].notna())]
elig = pd.Series(True, index=base.index)
for name, m in steps:
    elig &= m; print(f"after '{name}': {int(elig.sum())} / {n0}")
base = base[elig].copy()

def unit_ok(r):
    ref = DOSE_REF[r["drug_key"]]; u = norm_unit(r["dose_unit_normalized"])
    return u == ref["unit"] or (u in MASS and ref["unit"] in MASS)
base = base[base.apply(unit_ok, axis=1)]; print("after 'unit comparable with verified unit':", len(base))
if ALLOWED_ROUTES: base = base[base["route_normalized"].isin(ALLOWED_ROUTES)]; print("after route filter:", len(base))

base["unit_n"] = base["dose_unit_normalized"].map(norm_unit); base["dose_n"] = pd.to_numeric(base["dose_min"])
base = base.drop_duplicates(subset=["subject_id", "hadm_id", "drug_key", "dose_n", "unit_n", "route_normalized"]); print("after de-duplication:", len(base))

top = base["drug_key"].value_counts().head(MAX_DRUGS).index.tolist()
base = base[base["drug_key"].isin(top)]
rng0 = random.Random(SEED)
parts = []
for k, g in base.groupby("drug_key"):
    idx = list(g.index); rng0.shuffle(idx); parts.append(g.loc[sorted(idx[:MAX_ORIGINALS_PER_DRUG])])
ORIG = pd.concat(parts).sort_values(["subject_id", "hadm_id"]).reset_index(drop=True)
ORIG["original_id"] = [f"P{i:05d}" for i in range(len(ORIG))]
print(f"\nCURATED DRUGS ({len(top)}):", top)
print("originals:", len(ORIG), "| patients:", ORIG["subject_id"].nunique(), "| admissions:", ORIG["hadm_id"].nunique())
display(ORIG["drug_key"].value_counts().to_frame("originals"))

loaded /home/user9/SOURABH_S/data/canonical_medications.csv
canonical rows: 10256 | expected for the MIMIC-III Demo: about 10,256
after 'verification_ready': 10255 / 10256
after 'single numeric dose': 9468 / 10256
after 'matched to a verified drug with evidence': 1102 / 10256
after 'unit comparable with verified unit': 1102
after de-duplication: 527

CURATED DRUGS (8): ['potassium chloride (ORAL)', 'magnesium sulfate (INTRAVENOUS)', 'furosemide (INTRAVENOUS)', 'vancomycin (INTRAVENOUS)', 'metoprolol (INTRAVENOUS)', 'lorazepam (INTRAVENOUS)', 'furosemide (ORAL)', 'acetaminophen (INTRAVENOUS)']
originals: 527 | patients: 86 | admissions: 111


,originals
drug_key,
potassium chloride (ORAL),116
magnesium sulfate (INTRAVENOUS),113
furosemide (INTRAVENOUS),108
vancomycin (INTRAVENOUS),74
metoprolol (INTRAVENOUS),47
lorazepam (INTRAVENOUS),35
furosemide (ORAL),32
acetaminophen (INTRAVENOUS),2


In [23]:
# ===== 6. DETERMINISTIC LABEL LOGIC (dose / unit / frequency / condition / combined) =====
LV = {"SAFE": 0, "WARNING": 1, "HIGH": 2}; LN = ["SAFE", "WARNING", "HIGH"]
def worst(*s): return LN[max(LV[x] for x in s)]

def st_range(v, lo, hi):
    if v > hi * (1 + 1e-9): return "WARNING" if v / hi <= CHK["over_high"] * (1 + 1e-9) else "HIGH"
    if lo is not None and v < lo * (1 - 1e-9):
        r = lo / v if v > 0 else float("inf")
        return "WARNING" if r < CHK["under_high"] * (1 - 1e-9) else "HIGH"
    return "SAFE"
def st_over(x, lim):
    if x <= lim * (1 + 1e-9): return "SAFE"
    return "WARNING" if x / lim <= CHK["daily_high"] * (1 + 1e-9) else "HIGH"

def freq_rule(key):
    for k, v in FREQ_RULES.items():
        if k in key: return v, True
    return FREQ_DEFAULT, False

def label_dose(dose, unit, freq, ref, frule):
    # Returns dose/unit/frequency statuses for one prescription, mirroring Notebook 3's checker logic.
    unit = norm_unit(unit); ru = ref["unit"]; lo, hi, md = ref["lo"], ref["hi"], ref["max_daily"]
    _, fmin, fmax = frule
    ds, us, fs, v_eff = "SAFE", "SAFE", "SAFE", None
    if not unit:
        us = "WARNING"; ds = st_range(dose, lo, hi); v_eff = dose
    elif unit == ru:
        ds = st_range(dose, lo, hi); v_eff = dose
    elif unit in MASS and ru in MASS:
        conv = dose * MASS[unit] / MASS[ru]
        if st_range(conv, lo, hi) == "SAFE": v_eff = conv                       # legitimate alternative unit
        elif st_range(dose, lo, hi) == "SAFE":                                  # number fits the label unit => UNIT is the error
            gap = max(conv / hi, (lo / conv) if (lo and conv > 0) else 0)
            us = "HIGH" if gap >= CHK["unit_high"] else "WARNING"; v_eff = dose
        else: ds = st_range(conv, lo, hi); v_eff = conv
    else:
        us = "WARNING"
    pd_ = FREQ_PER_DAY[freq]
    if pd_ > fmax: fs = st_over(pd_, fmax)
    elif pd_ < fmin and fmin / pd_ >= 2.0: fs = "WARNING"
    if md is not None and ds == "SAFE" and v_eff is not None: fs = worst(fs, st_over(v_eff * pd_, md))
    return ds, us, fs

_NEG = re.compile(r"\bnot contraindicated\b|\bno contraindications?\b|\bno known contraindications?\b")
def condition_hits(cond, key):
    # (level, snippet_id, snippet_text) for one condition against the VISIBLE snippets of a drug.
    syn = COND[cond]["syn"]
    for sec, lvl, letter in (("contra", "HIGH", "C"), ("warnings", "WARNING", "W")):
        for i, t in enumerate(VISIBLE[key][sec], 1):
            tl = t.lower()
            if _NEG.search(tl): continue
            if any(s in tl for s in syn): return lvl, f"{letter}{i}", t
    return "SAFE", None, None

def label_conditions(conds, key):
    best = ("SAFE", None, None, None)
    for c in conds:
        lvl, sid, txt = condition_hits(c, key)
        if LV[lvl] > LV[best[0]]: best = (lvl, sid, txt, c)
    return best

def combine(dose_risk, cond_risk):
    # 'both_required' mode (Notebook 3 default): raised only when a dose problem AND a condition risk are both present; escalated one level.
    if dose_risk != "SAFE" and cond_risk != "SAFE": return LN[min(2, max(LV[dose_risk], LV[cond_risk]) + 1)]
    return "SAFE"

# quick self-test on the first verified drug
_k = next(iter(DOSE_REF)); _r = DOSE_REF[_k]; _fr, _ = freq_rule(_k)
_mid = (_r["lo"] + _r["hi"]) / 2 if _r["lo"] else _r["hi"] / 2
assert label_dose(_mid, _r["unit"], _fr[0], _r, _fr)[0] == "SAFE", "a mid-range dose should be SAFE"
assert label_dose(_r["hi"] * 10, _r["unit"], _fr[0], _r, _fr)[0] == "HIGH", "10x max should be HIGH"
print("label logic self-test passed for", _k)

label logic self-test passed for furosemide (ORAL)


In [24]:
# ===== 7. SYNTHETIC ERROR GENERATOR =====
def clean_num(x):
    x = round(float(x), 4); return int(x) if float(x).is_integer() else x

SCALE = {"dose_x2": 2.0, "dose_x5": 5.0, "dose_x10": 10.0, "dose_div2": 0.5, "dose_div10": 0.1}
ERROR_TYPES = list(SCALE) + ["unit_change", "frequency_change", "digit_transposition", "dropped_digit", "missing_unit"]

def perturb(et, dose, unit, freq, rng):
    # Returns (dose, unit, freq) after injecting one error, or None if the error is not applicable.
    if et in SCALE: return clean_num(dose * SCALE[et]), unit, freq
    if et == "unit_change":
        if unit not in MASS: return None
        return dose, rng.choice([u for u in MASS if u != unit]), freq
    if et == "frequency_change": return dose, unit, rng.choice([f for f in FREQ_PER_DAY if f != freq])
    if et == "missing_unit": return dose, "", freq
    if float(dose).is_integer() and int(dose) >= 10:
        s = str(int(dose))
        if et == "digit_transposition":
            pos = [i for i in range(len(s) - 1) if s[i] != s[i + 1]]
            if not pos: return None
            i = rng.choice(pos); t = s[:i] + s[i + 1] + s[i] + s[i + 2:]
            return (clean_num(int(t)), unit, freq) if t[0] != "0" else None
        if et == "dropped_digit":
            i = rng.randrange(len(s)); t = s[:i] + s[i + 1:]
            return (clean_num(int(t)), unit, freq) if t and t[0] != "0" and int(t) > 0 else None
    return None

def pick_balanced(applicable, counts, k, rng):
    # choose k distinct error types, always preferring the globally least-used ones (keeps error types balanced)
    pool, chosen = list(applicable), []
    for _ in range(min(k, len(pool))):
        m = min(counts[e] for e in pool); e = rng.choice([x for x in pool if counts[x] == m])
        chosen.append(e); pool.remove(e); counts[e] += 1
    return chosen

def fmt_rx(drug, dose, unit, freq): return f"{drug} {dose} {unit} {freq}".replace("  ", " ").strip()
print("error types:", ERROR_TYPES)

error types: ['dose_x2', 'dose_x5', 'dose_x10', 'dose_div2', 'dose_div10', 'unit_change', 'frequency_change', 'digit_transposition', 'dropped_digit', 'missing_unit']


In [25]:
# ===== 8. BUILD THE FULL LABELLED DATASET =====
rng = random.Random(SEED)
counts = Counter(); rows = []; defaults_used = set()
all_cond_names = list(COND)

for _, o in ORIG.iterrows():
    key = o["drug_key"]; ref = DOSE_REF[key]; frule, specific = freq_rule(key)
    if not specific: defaults_used.add(key)
    std_f = frule[0]; unit0 = o["unit_n"]; dose0 = clean_num(o["dose_n"])
    real = REAL_CONDS.get(o["hadm_id"], [])
    relevant = [c for c in all_cond_names if condition_hits(c, key)[0] != "SAFE"]
    irrelevant = [c for c in all_cond_names if c not in relevant]
    contexts = [("real", list(real))]
    for j in range(N_SYNTHETIC_CONTEXTS):
        pool = relevant if (j == 0 and relevant) else irrelevant
        if pool: contexts.append(("synthetic", [rng.choice(pool)]))
    applicable = {}
    for et in ERROR_TYPES:
        p = perturb(et, dose0, unit0, std_f, rng)
        if p is not None and p != (dose0, unit0, std_f): applicable[et] = p
    chosen = pick_balanced(list(applicable), counts, VARIANTS_PER_ORIGINAL, rng)
    for vi, et in enumerate(["none"] + chosen):
        d, u, f = (dose0, unit0, std_f) if et == "none" else applicable[et]
        ds, us, fs = label_dose(d, u, f, ref, frule)
        src, conds = rng.choice(contexts)
        cl, sid, stxt, mcond = label_conditions(conds, key)
        dose_risk = worst(ds, us, fs); comb = combine(dose_risk, cl)
        labels = dict(dose_status=ds, unit_status=us, frequency_status=fs, condition_risk=cl, combined_risk=comb)
        rows.append(dict(
            example_id=f"{o['original_id']}_v{vi}", original_id=o["original_id"], subject_id=int(o["subject_id"]), hadm_id=int(o["hadm_id"]),
            drug=key, drug_raw=o["drug_name_normalized"], route=o["route_normalized"],
            dose=d, unit=u, frequency=f, conditions="; ".join(conds), condition_source=src, error_type=et,
            original_prescription=fmt_rx(key, dose0, unit0, std_f), modified_prescription=fmt_rx(key, d, u, f),
            original_dose=dose0, original_unit=unit0, original_frequency=std_f,
            **labels, expected_result=json.dumps(labels), matched_condition=mcond, matched_evidence_id=sid, matched_warning=stxt))

DATA = pd.DataFrame(rows)
print("total examples:", len(DATA), "| originals:", DATA["original_id"].nunique())
if defaults_used: print("⚠️ these drugs use FREQ_DEFAULT (edit FREQ_RULES, then re-run):", sorted(defaults_used))
print("\nerror types:"); display(DATA["error_type"].value_counts().to_frame("n"))
print("label distributions:")
display(pd.DataFrame({c: DATA[c].value_counts() for c in ["dose_status", "unit_status", "frequency_status", "condition_risk", "combined_risk"]}).fillna(0).astype(int))
print("condition source:", dict(DATA["condition_source"].value_counts()))

total examples: 2635 | originals: 527
⚠️ these drugs use FREQ_DEFAULT (edit FREQ_RULES, then re-run): ['magnesium sulfate (INTRAVENOUS)', 'potassium chloride (ORAL)', 'vancomycin (INTRAVENOUS)']

error types:


,n
error_type,
none,527
dose_x5,239
missing_unit,239
frequency_change,239
dose_x2,239
dose_div10,239
unit_change,239
dose_div2,238
dose_x10,238


label distributions:


,dose_status,unit_status,frequency_status,condition_risk,combined_risk
HIGH,849,188,90,610,886
SAFE,1368,2208,2428,1337,1749
WARNING,418,239,117,688,0


condition source: {'synthetic': np.int64(1778), 'real': np.int64(857)}


In [26]:
# ===== 9. GROUPED SPLIT BY PATIENT (no patient / admission / original prescription in two splits) =====
sizes = DATA.groupby("subject_id")["original_id"].nunique().to_dict()
subs = sorted(sizes); random.Random(SEED).shuffle(subs)
total = sum(sizes.values()); target = {s: SPLIT_FRACTIONS[s] * total for s in SPLIT_FRACTIONS}
cur = {s: 0 for s in SPLIT_FRACTIONS}; assign = {}
for sub in subs:
    s = max(SPLIT_FRACTIONS, key=lambda x: target[x] - cur[x]); assign[sub] = s; cur[s] += sizes[sub]
DATA["split"] = DATA["subject_id"].map(assign)
SPL = {s: DATA[DATA["split"] == s].drop(columns=["split"]).reset_index(drop=True) for s in SPLIT_FRACTIONS}

# ---- leakage checks (hard failures) ----
for col in ["subject_id", "hadm_id", "original_id"]:
    for a, b in itertools.combinations(SPL, 2):
        shared = set(SPL[a][col]) & set(SPL[b][col])
        assert not shared, f"LEAKAGE: {col} shared between {a} and {b}: {list(shared)[:5]}"
assert DATA["example_id"].is_unique
print("✅ no leakage by subject_id / hadm_id / original_id")

CATS = ["dose_status", "unit_status", "frequency_status", "condition_risk", "combined_risk"]
rep = []
for s, d in SPL.items():
    row = dict(split=s, examples=len(d), originals=d["original_id"].nunique(), patients=d["subject_id"].nunique())
    for c in CATS: row[f"{c}_positive"] = int((d[c] != "SAFE").sum())
    rep.append(row)
rep = pd.DataFrame(rep); display(rep)
for s, d in SPL.items():
    for c in CATS:
        if (d[c] != "SAFE").sum() < 10: print(f"⚠️ {s}: only {(d[c] != 'SAFE').sum()} positive examples for {c} -> its metrics will be very noisy")
print("\nerror types per split:"); display(pd.crosstab(DATA["error_type"], DATA["split"]))

✅ no leakage by subject_id / hadm_id / original_id


,split,examples,originals,patients,dose_status_positive,unit_status_positive,frequency_status_positive,condition_risk_positive,combined_risk_positive
0,train,1870,374,52,900,304,150,924,629
1,val,380,76,16,185,60,27,209,141
2,test,385,77,18,182,63,30,165,116



error types per split:


split,test,train,val
error_type,,,
digit_transposition,3,5,2
dose_div10,35,171,33
dose_div2,37,165,36
dose_x10,36,169,33
dose_x2,33,169,37
dose_x5,33,172,34
dropped_digit,27,133,28
frequency_change,34,171,34
missing_unit,35,172,32


In [27]:
# ===== 10. SAVE (files Notebook 3 reads) =====
OUT_DIR.mkdir(parents=True, exist_ok=True)
for s, d in SPL.items(): d.to_csv(OUT_DIR / f"{s}.csv", index=False)
pd.DataFrame([dict(drug=k, dosage="\n".join(EVIDENCE[k]["dosage"].split("\n")), warnings=EVIDENCE[k]["warnings"], contraindications=EVIDENCE[k]["contra"])
              for k in sorted(ORIG["drug_key"].unique())]).to_csv(OUT_DIR / "evidence.csv", index=False)
DATA.sample(min(40, len(DATA)), random_state=SEED).to_csv(OUT_DIR / "examples_preview.csv", index=False)
report = dict(seed=SEED, curated_drugs=sorted(ORIG["drug_key"].unique()), n_examples={s: len(d) for s, d in SPL.items()},
              variants_per_original=VARIANTS_PER_ORIGINAL, n_synthetic_contexts=N_SYNTHETIC_CONTEXTS, thresholds=CHK,
              snippet_caps=dict(chars=SNIPPET_CHARS, max=MAX_SNIPPETS, section=SECTION_CHARS), split_unit="subject_id",
              freq_defaults_used=sorted(defaults_used), error_types=ERROR_TYPES,
              notes=["Frequency is not in MIMIC; standard frequency assigned from FREQ_RULES",
                     "Dose/unit/frequency labels from verified dose_table.json + FREQ_RULES",
                     "Condition labels from visible openFDA snippets; contexts are real (ICD-9) + synthetic",
                     "Original MIMIC prescriptions are labelled by the same rules (not assumed safe)"])
(OUT_DIR / "split_report.json").write_text(json.dumps(report, indent=1))
print("saved to", OUT_DIR); [print(" -", p.name, f"({p.stat().st_size/1e3:.0f} KB)") for p in sorted(OUT_DIR.iterdir())]
print(f'''
=== Paste into Notebook 3, section 2 ===
TRAIN_PATH    = "{OUT_DIR / 'train.csv'}"
VAL_PATH      = "{OUT_DIR / 'val.csv'}"
TEST_PATH     = "{OUT_DIR / 'test.csv'}"
EVIDENCE_PATH = "{OUT_DIR / 'evidence.csv'}"
COLS: defaults already match (example_id, original_id, drug, dose, unit, frequency, conditions, error_type,
      dose_status, unit_status, frequency_status, condition_risk, combined_risk).
Leave range_min/range_max/max_daily = None (keeps the rules-only baseline independent of the label source).
''')

saved to /home/user9/SOURABH_S/data/splits
 - evidence.csv (3468 KB)
 - examples_preview.csv (20 KB)
 - split_report.json (1 KB)
 - test.csv (190 KB)
 - train.csv (955 KB)
 - val.csv (204 KB)

=== Paste into Notebook 3, section 2 ===
TRAIN_PATH    = "/home/user9/SOURABH_S/data/splits/train.csv"
VAL_PATH      = "/home/user9/SOURABH_S/data/splits/val.csv"
TEST_PATH     = "/home/user9/SOURABH_S/data/splits/test.csv"
EVIDENCE_PATH = "/home/user9/SOURABH_S/data/splits/evidence.csv"
COLS: defaults already match (example_id, original_id, drug, dose, unit, frequency, conditions, error_type,
      dose_status, unit_status, frequency_status, condition_risk, combined_risk).
Leave range_min/range_max/max_daily = None (keeps the rules-only baseline independent of the label source).



In [28]:
import pandas as pd
d = pd.read_csv("/home/user9/SOURABH_S/data/splits/test.csv")
print(d.shape, "| patients:", d["subject_id"].nunique())
print(d[["drug","dose","unit","frequency","conditions","error_type","dose_status","unit_status","frequency_status","condition_risk","combined_risk"]].head(8).to_string())
print({c: int((d[c] != "SAFE").sum()) for c in ["dose_status","unit_status","frequency_status","condition_risk","combined_risk"]})

(385, 27) | patients: 18
                       drug    dose unit frequency        conditions     error_type dose_status unit_status frequency_status condition_risk combined_risk
0  metoprolol (INTRAVENOUS)     5.0   mg       BID      hyperkalemia           none        SAFE        SAFE             SAFE           SAFE          SAFE
1  metoprolol (INTRAVENOUS)    50.0   mg       BID      hypertension       dose_x10        HIGH        SAFE             SAFE           SAFE          SAFE
2  metoprolol (INTRAVENOUS)     2.5   mg       BID     heart failure      dose_div2        SAFE        SAFE             SAFE           HIGH          SAFE
3  metoprolol (INTRAVENOUS)     5.0  mcg       BID      hyperkalemia    unit_change        SAFE        SAFE             SAFE           SAFE          SAFE
4  metoprolol (INTRAVENOUS)    10.0   mg       BID     heart failure        dose_x2     WARNING        SAFE             SAFE           HIGH          HIGH
5  vancomycin (INTRAVENOUS)  1000.0   mg       BID 